# Etiquetas de medicamentos: RAG con filtro por campo

**Data Science con Python · Universidad del Pacífico · 2026-II**
Jefe de práctica: Paul Melo Ramos · Profesor: Alexander Quispe

---

## Por qué este corpus es distinto a todos los anteriores

Una etiqueta de medicamento **no es un texto: es un formulario**.

```
indications_and_usage      ¿para qué sirve?
warnings                   ¿qué cuidados hay?
dosage_and_administration  ¿cuánto se toma?
drug_interactions          ¿con qué no se mezcla?
contraindications          ¿quién no debe tomarlo?
adverse_reactions          ¿qué efectos adversos tiene?
pregnancy                  ¿y en el embarazo?
```

El documento **ya viene troceado por quien lo escribió**. Eso cambia todo el pipeline: el
chunking deja de ser una decisión sobre cuántos caracteres cortar, y pasa a ser una decisión
sobre **qué campo buscar**.

Y abre una funcionalidad que no teníamos: una pregunta sobre interacciones debe buscar
**solo** en los chunks de interacciones. Filtro por metadata, en acción.

## El pipeline

```
OFFLINE (celdas 3 a 6)
  API openFDA ──▶ JSON crudo ──▶ un chunk POR CAMPO ──▶ vectores ──▶ ChromaDB

ONLINE (celdas 7 a 10)
  pregunta ──▶ router de campo ──▶ búsqueda filtrada ──▶ ¿supera el umbral?
                                                           ├─ no ──▶ "no sé"
                                                           └─ sí ──▶ DeepSeek ──▶ respuesta + cita
```

## Las 10 celdas

| # | Qué hace | El número que sale |
|---|---|---|
| 1 | esta portada | — |
| 2 | dependencias, configuración y llaves | 40 principios activos |
| 3 | descarga con caché desde openFDA | etiquetas obtenidas y fallidas |
| 4 | **informe de cobertura de campos** | qué % de etiquetas trae cada sección |
| 5 | chunking por campo con metadata | chunks por tipo |
| 6 | embeddings locales e índice | segundos de indexado |
| 7 | **búsqueda con filtro por campo** | con filtro contra sin filtro |
| 8 | generación con DeepSeek | USD por consulta |
| 9 | evaluación: Recall@k global y por campo | dónde es débil el sistema |
| 10 | demo de tres preguntas y cierre | costo de la sesión |

---

## Aviso de fuente, y va en serio

Este sistema lee **etiquetas de medicamentos publicadas por la FDA de Estados Unidos**, en
inglés, derivadas de documentos SPL que presentan los fabricantes. Son información de
referencia técnica.

**No corresponden al registro sanitario peruano, no sustituyen la indicación de un profesional
de la salud, y no son la fuente correcta para decidir nada aquí.**

Que el sistema cite su fuente **y sus límites** es parte de lo que se evalúa.

Los datos de openFDA son obra del gobierno de Estados Unidos, libres de copyright allá, y
deben citarse como openFDA.

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 2 — SETUP: dependencias, configuración y llaves
# ═══════════════════════════════════════════════════════════════════════════
import importlib.util, subprocess, sys, os, re, time, json, csv, urllib.parse, urllib.request
from pathlib import Path
from collections import Counter, defaultdict

REQ = [("chromadb", "chromadb"),
       ("sentence-transformers", "sentence_transformers"),
       ("openai", "openai"),
       ("requests", "requests")]
faltan = [p for p, m in REQ if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan]).returncode
    if rc != 0:
        rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                             "--break-system-packages", *faltan]).returncode
    if rc != 0:
        print("\nNo pude instalar solo. Crea un entorno virtual y corre:")
        print("  pip install " + " ".join(faltan))
        raise SystemExit("Instalacion pendiente")

import requests
import numpy as np

# --- Configuración única. Si un número aparece más abajo, está mal. --------
CONFIG = {
    "api_url": "https://api.fda.gov/drug/label.json",
    "etiquetas_por_dci": 2,        # cuántas etiquetas traer por principio activo
    "pausa_s": 0.35,               # cortesía entre peticiones
    "dir_crudo": "data/raw_openfda",

    # 40 principios activos de uso frecuente en el Perú.
    # Cámbialos por los de tu dominio: esta lista es el corpus.
    "dcis": [
        "ibuprofen", "acetaminophen", "naproxen", "aspirin", "diclofenac",
        "metformin", "glibenclamide", "insulin human", "atorvastatin", "simvastatin",
        "losartan", "enalapril", "amlodipine", "captopril", "furosemide",
        "omeprazole", "ranitidine", "metoclopramide", "loperamide", "dimenhydrinate",
        "amoxicillin", "azithromycin", "ciprofloxacin", "cephalexin", "clindamycin",
        "salbutamol", "albuterol", "loratadine", "cetirizine", "prednisone",
        "sertraline", "fluoxetine", "alprazolam", "carbamazepine", "levothyroxine",
        "warfarin", "clopidogrel", "tramadol", "ketorolac", "ondansetron",
    ],

    # Las secciones que nos interesan. El informe de cobertura dirá cuáles existen.
    "campos": [
        "indications_and_usage", "warnings", "dosage_and_administration",
        "drug_interactions", "contraindications", "adverse_reactions",
        "pregnancy", "boxed_warning",
    ],

    # Router: una regla puede apuntar a VARIOS campos.
    # Por que varios: el aviso sobre alcohol vive en "warnings" en las etiquetas
    # OTC y en "drug_interactions" en las de receta. Un router de un solo campo
    # falla en la mitad de los casos, y el fallo es invisible.
    "router": [
        {"campos": ["drug_interactions", "warnings"],
         "claves": ["alcohol", "interac", "junto con", "al mismo tiempo",
                    "mezclar", "combinar", "con otro medicamento", "tomar con"]},
        {"campos": ["pregnancy", "warnings"],
         "claves": ["embaraz", "gestaci", "lactancia", "amamant", "dando de lactar"]},
        {"campos": ["dosage_and_administration"],
         "claves": ["dosis", "cuanto tomo", "cuánto tomo", "cada cuantas",
                    "cada cuántas", "posolog", "cuantas pastillas"]},
        {"campos": ["contraindications", "warnings"],
         "claves": ["no debe", "contraindic", "quien no puede", "quién no puede"]},
        {"campos": ["adverse_reactions"],
         # Raices, no frases completas: "efecto secundario" NO coincide con
         # "efectos secundarios" por la ese. Es el bug mas tonto y mas comun
         # de un router por palabras clave.
         "claves": ["secundari", "advers", "reaccion", "reacción", "colateral",
                    "me hace mal", "me cae mal", "da sueño", "da sueno"]},
        {"campos": ["warnings"],
         "claves": ["advertenc", "cuidado", "riesgo", "precaucion", "precaución"]},
        {"campos": ["indications_and_usage"],
         "claves": ["sirve para", "para que", "para qué", "indicad", "trata"]},
    ],

    "chunk_max": 1200,
    "chunk_solape": 180,

    "modelo_emb": "intfloat/multilingual-e5-small",
    "prefijo_doc": "passage: ",
    "prefijo_query": "query: ",
    "lote": 64,

    "chroma_path": "chroma_openfda",
    "coleccion": "openfda_labels",

    "base_url": "https://api.deepseek.com",
    "modelo_llm": "deepseek-flash",
    "temperature": 0.1,
    "max_tokens": 500,
    "k": 5,
    "umbral_similitud": 0.78,

    "precios": {
        "verificado_el": "2026-09-14",
        "fuente": "https://api-docs.deepseek.com/quick_start/pricing",
        "in": 0.15, "in_cache": 0.003, "out": 0.60,
    },
}

Path(CONFIG["dir_crudo"]).mkdir(parents=True, exist_ok=True)
# Cargar .env si existe (así funciona igual que el proyecto de Beca 18)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv
    load_dotenv()


# --- Llaves. Nunca escritas dentro del código. -----------------------------
def llave(nombre, obligatoria=True):
    try:
        from google.colab import userdata
        v = userdata.get(nombre)
        if v:
            return v
    except Exception:
        pass
    v = os.getenv(nombre)
    if v:
        return v
    if obligatoria:
        from getpass import getpass
        return getpass(f"{nombre}: ").strip() or None
    return None

# openFDA funciona SIN llave (con límite bajo). Con llave: 240/min y 120,000/día.
FDA_KEY = llave("FDA_API_KEY")
DEEPSEEK_KEY = llave("DEEPSEEK_API_KEY")

print(f"Principios activos en el corpus : {len(CONFIG['dcis'])}")
print(f"Campos a extraer                : {len(CONFIG['campos'])}")
print(f"Llave openFDA (opcional)        : {bool(FDA_KEY)}")
print(f"Llave DeepSeek (celda 8 en adel.): {bool(DEEPSEEK_KEY)}")

Principios activos en el corpus : 40
Campos a extraer                : 8
Llave openFDA (opcional)        : True
Llave DeepSeek (celda 8 en adel.): True


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 3 — DESCARGA CON CACHÉ
#
#  Regla del proyecto: el JSON CRUDO se guarda tal cual y nunca se modifica.
#  Toda transformación parte del crudo. Si mañana cambias el chunking, no
#  vuelves a pedirle nada a la FDA.
#
#  openFDA devuelve 404 cuando NO hay coincidencias. No es un error de red:
#  es la forma en que dice "ese medicamento no está". Hay que distinguirlos.
# ═══════════════════════════════════════════════════════════════════════════

def bajar_etiquetas(dci, limite=None):
    """Devuelve la lista de etiquetas de un principio activo, usando caché."""
    limite = limite or CONFIG["etiquetas_por_dci"]
    destino = Path(CONFIG["dir_crudo"]) / f"{dci.replace(' ', '_')}.json"

    if destino.exists():                                  # caché
        return json.loads(destino.read_text(encoding="utf-8")), "cache"

    params = {"search": f'openfda.generic_name:"{dci}"', "limit": limite}
    if FDA_KEY:
        params["api_key"] = FDA_KEY
    url = CONFIG["api_url"] + "?" + urllib.parse.urlencode(params)

    try:
        r = requests.get(url, timeout=40)
        if r.status_code == 404:
            return [], "sin_coincidencias"
        r.raise_for_status()
        resultados = r.json().get("results", [])
        destino.write_text(json.dumps(resultados, ensure_ascii=False),
                           encoding="utf-8")
        return resultados, "descargado"
    except Exception as e:
        return [], f"error: {str(e)[:60]}"


etiquetas, bitacora = {}, []
t0 = time.time()
for i, dci in enumerate(CONFIG["dcis"], 1):
    res, estado = bajar_etiquetas(dci)
    etiquetas[dci] = res
    bitacora.append({"dci": dci, "etiquetas": len(res), "estado": estado})
    print(f"  [{i:>2}/{len(CONFIG['dcis'])}] {dci:<18} {len(res)} etiqueta(s)  ({estado})")
    if estado == "descargado":
        time.sleep(CONFIG["pausa_s"])

conteo = Counter(b["estado"].split(":")[0] for b in bitacora)
total_etq = sum(b["etiquetas"] for b in bitacora)

print(f"\nBITÁCORA DE DESCARGA  ({time.time()-t0:.1f} s)")
for estado, n in conteo.items():
    print(f"  {estado:<18} {n}")
print(f"  etiquetas totales  {total_etq}")

sin_datos = [b["dci"] for b in bitacora if b["etiquetas"] == 0]
if sin_datos:
    print(f"\n  Sin coincidencias en openFDA: {sin_datos}")
    print("""
  Esto NO es un fallo del código. Es un hallazgo sobre el corpus: hay
  principios activos de uso común en el Perú que no tienen etiqueta en la
  FDA, o que allá se registran con otro nombre (salbutamol -> albuterol).
  Anótalo: es la primera limitación del informe.""")

  [ 1/40] ibuprofen          2 etiqueta(s)  (cache)
  [ 2/40] acetaminophen      2 etiqueta(s)  (cache)
  [ 3/40] naproxen           2 etiqueta(s)  (cache)
  [ 4/40] aspirin            2 etiqueta(s)  (cache)
  [ 5/40] diclofenac         2 etiqueta(s)  (cache)
  [ 6/40] metformin          2 etiqueta(s)  (cache)
  [ 7/40] glibenclamide      0 etiqueta(s)  (sin_coincidencias)
  [ 8/40] insulin human      2 etiqueta(s)  (cache)
  [ 9/40] atorvastatin       2 etiqueta(s)  (cache)
  [10/40] simvastatin        2 etiqueta(s)  (cache)
  [11/40] losartan           2 etiqueta(s)  (cache)
  [12/40] enalapril          2 etiqueta(s)  (cache)
  [13/40] amlodipine         2 etiqueta(s)  (cache)
  [14/40] captopril          2 etiqueta(s)  (cache)
  [15/40] furosemide         2 etiqueta(s)  (cache)
  [16/40] omeprazole         2 etiqueta(s)  (cache)
  [17/40] ranitidine         2 etiqueta(s)  (cache)
  [18/40] metoclopramide     2 etiqueta(s)  (cache)
  [19/40] loperamide         2 etiqueta(s)  (cache)


In [17]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 4 — INFORME DE COBERTURA DE CAMPOS
#
#  Esta celda es el corazón del proyecto y va ANTES de chunkear.
#
#  No describe el corpus: DEFINE qué preguntas el sistema tiene derecho a
#  contestar. Si el 65% de las etiquetas no dice nada sobre embarazo, un
#  sistema que siempre responde sobre embarazo está inventando.
# ═══════════════════════════════════════════════════════════════════════════

cobertura = {c: 0 for c in CONFIG["campos"]}
largos = defaultdict(list)
etiquetas_utiles = 0
campos_vistos = Counter()

for dci, lista in etiquetas.items():
    for etq in lista:
        etiquetas_utiles += 1
        campos_vistos.update(k for k in etq.keys() if k != "openfda")
        for campo in CONFIG["campos"]:
            valor = etq.get(campo)
            if valor:
                texto = " ".join(valor) if isinstance(valor, list) else str(valor)
                if texto.strip():
                    cobertura[campo] += 1
                    largos[campo].append(len(texto))

print(f"Etiquetas analizadas: {etiquetas_utiles}\n")
print(f"{'campo':<30}{'presente':>10}{'%':>8}{'largo medio':>14}")
print("-" * 62)
for campo in CONFIG["campos"]:
    n = cobertura[campo]
    pct = n / etiquetas_utiles * 100 if etiquetas_utiles else 0
    lm = round(sum(largos[campo]) / len(largos[campo])) if largos[campo] else 0
    alerta = "   <- escaso" if pct < 50 else ""
    print(f"{campo:<30}{n:>10}{pct:>7.0f}%{lm:>14,}{alerta}")

print("""
  CÓMO SE LEE ESTA TABLA

  Cada fila con cobertura baja es una pregunta que el sistema NO puede
  responder de forma confiable. No porque el modelo falle: porque el dato
  no está.

  Un sistema honesto se abstiene ahí. Uno mal construido inventa, y suena
  igual de seguro en los dos casos.

  Esta tabla va en tu informe y en tu video. Es la diferencia entre
  "construí un chatbot" y "sé qué puede y qué no puede mi chatbot".
""")

print("\nOtros campos que trae la API y que NO estamos usando (top 12):")
extra = [(k, v) for k, v in campos_vistos.most_common(40)
         if k not in CONFIG["campos"]][:12]
for k, v in extra:
    print(f"  {k:<42} en {v} etiquetas")
print("\n  Amplía CONFIG['campos'] si alguno te sirve. Es una línea de config.")

Etiquetas analizadas: 76

campo                           presente       %   largo medio
--------------------------------------------------------------
indications_and_usage                 76    100%         1,449
warnings                              36     47%         3,690   <- escaso
dosage_and_administration             76    100%         4,293
drug_interactions                     51     67%         5,330
contraindications                     56     74%           705
adverse_reactions                     56     74%         5,964
pregnancy                             46     61%         2,592
boxed_warning                         29     38%         1,808   <- escaso

  CÓMO SE LEE ESTA TABLA

  Cada fila con cobertura baja es una pregunta que el sistema NO puede
  responder de forma confiable. No porque el modelo falle: porque el dato
  no está.

  Un sistema honesto se abstiene ahí. Uno mal construido inventa, y suena
  igual de seguro en los dos casos.

  Esta tabla va en tu inf

In [18]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 5 — CHUNKING POR CAMPO
#
#  Aquí el documento YA viene troceado: cada sección es un chunk natural.
#  Lo único que hacemos nosotros es subdividir las secciones muy largas.
#
#  Lo importante es la METADATA: dci, campo, marca y fabricante. Sin el
#  campo en metadata, la celda 7 no existe.
# ═══════════════════════════════════════════════════════════════════════════

def partir(texto, maximo, solape):
    """Ventana deslizante solo para secciones que pasan el máximo."""
    if len(texto) <= maximo:
        return [texto]
    pedazos, i = [], 0
    while i < len(texto):
        pedazos.append(texto[i:i + maximo])
        i += maximo - solape
    return pedazos


def primer(valor):
    if isinstance(valor, list):
        return valor[0] if valor else ""
    return str(valor or "")


chunks = []
for dci, lista in etiquetas.items():
    for n_etq, etq in enumerate(lista):
        of = etq.get("openfda", {})
        marca = primer(of.get("brand_name"))
        fabricante = primer(of.get("manufacturer_name"))
        tipo = primer(of.get("product_type"))

        for campo in CONFIG["campos"]:
            valor = etq.get(campo)
            if not valor:
                continue
            texto = " ".join(valor) if isinstance(valor, list) else str(valor)
            texto = re.sub(r"\s+", " ", texto).strip()
            if not texto:
                continue

            for j, pedazo in enumerate(partir(texto, CONFIG["chunk_max"],
                                              CONFIG["chunk_solape"])):
                chunks.append({
                    "id": f"{dci.replace(' ','_')}__{campo}__{n_etq}_{j}",
                    "texto": pedazo,
                    "meta": {"dci": dci, "campo": campo, "marca": marca,
                             "fabricante": fabricante, "tipo": tipo,
                             "fuente": "openFDA drug/label"},
                })

por_campo = Counter(c["meta"]["campo"] for c in chunks)
por_dci = Counter(c["meta"]["dci"] for c in chunks)

print(f"Chunks totales: {len(chunks):,}\n")
print(f"{'campo':<30}{'chunks':>9}")
print("-" * 39)
for campo, n in por_campo.most_common():
    print(f"{campo:<30}{n:>9,}")

print(f"\nPrincipios activos con chunks: {len(por_dci)}")
print(f"Chunks por principio activo: min {min(por_dci.values())}, "
      f"máx {max(por_dci.values())}")

c = chunks[0]
print(f"\nEjemplo -> {c['id']}")
print(f"  campo: {c['meta']['campo']} | marca: {c['meta']['marca']}")
print(f"  {c['texto'][:220]}...")

Chunks totales: 1,572

campo                            chunks
---------------------------------------
dosage_and_administration           357
adverse_reactions                   356
drug_interactions                   292
indications_and_usage               149
warnings                            146
pregnancy                           140
contraindications                    68
boxed_warning                        64

Principios activos con chunks: 38
Chunks por principio activo: min 6, máx 108

Ejemplo -> ibuprofen__indications_and_usage__0_0
  campo: indications_and_usage | marca: Ibuprofen Dye Free
  Uses temporarily relieves minor aches and pains due to: headache toothache backache menstrual cramps the common cold muscular aches minor pain of arthritis temporarily reduces fever...


In [19]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 6 — EMBEDDINGS LOCALES E ÍNDICE
#
#  Embeddings locales: sin API, sin límites, sin costo. Eso es lo que hace
#  posible reindexar diez veces mientras afinas, en vez de una sola.
#
#  Los prefijos "query:" y "passage:" son obligatorios en la familia e5.
#  Omitirlos no da error: solo empeora la recuperación, en silencio.
# ═══════════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer
import chromadb

print(f"Cargando {CONFIG['modelo_emb']} (la primera vez descarga ~470 MB)...")
modelo_emb = SentenceTransformer(CONFIG["modelo_emb"])
DIM = modelo_emb.get_sentence_embedding_dimension()
print(f"Dimensiones: {DIM}")


def vec_docs(textos):
    return modelo_emb.encode([CONFIG["prefijo_doc"] + t for t in textos],
                             batch_size=CONFIG["lote"], normalize_embeddings=True,
                             show_progress_bar=len(textos) > 200).tolist()


def vec_query(texto):
    return modelo_emb.encode([CONFIG["prefijo_query"] + texto],
                             normalize_embeddings=True)[0].tolist()


cliente = chromadb.PersistentClient(path=CONFIG["chroma_path"])
col = cliente.get_or_create_collection(name=CONFIG["coleccion"],
                                       metadata={"hnsw:space": "cosine"})

ya = col.count()
if ya < len(chunks):
    if ya:
        print(f"Reanudando desde el chunk {ya}...")
    t0 = time.time()
    for i in range(ya, len(chunks), CONFIG["lote"]):
        bloque = chunks[i:i + CONFIG["lote"]]
        textos = [c["texto"] for c in bloque]
        col.add(ids=[c["id"] for c in bloque], documents=textos,
                embeddings=vec_docs(textos), metadatas=[c["meta"] for c in bloque])
    print(f"Indexado en {time.time()-t0:.1f} segundos, costo USD 0.00")
else:
    print("El índice ya estaba completo: no se reindexa nada. Eso es idempotencia.")

print(f"Chunks en el índice: {col.count():,}")
print("\nRecordatorio: Chroma devuelve DISTANCIA, no similitud.")
print("  similitud = 1 - distancia")

Cargando intfloat/multilingual-e5-small (la primera vez descarga ~470 MB)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2035.18it/s]


Dimensiones: 384
El índice ya estaba completo: no se reindexa nada. Eso es idempotencia.
Chunks en el índice: 1,572

Recordatorio: Chroma devuelve DISTANCIA, no similitud.
  similitud = 1 - distancia


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22756\2656746620.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = modelo_emb.get_sentence_embedding_dimension()


In [20]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 7 — BÚSQUEDA CON FILTRO POR CAMPO
#
#  La funcionalidad nueva de este proyecto. Una pregunta sobre interacciones
#  debe buscar SOLO en chunks de interacciones. La metadata no es decoración:
#  convierte una búsqueda en siete búsquedas distintas.
# ═══════════════════════════════════════════════════════════════════════════

def detectar_campos(pregunta):
    """Router por palabras clave. Devuelve una LISTA de campos, o None.

    La primera regla que coincide gana. Si ninguna coincide, no se filtra:
    mejor buscar en todo que filtrar hacia el campo equivocado.
    """
    p = pregunta.lower()
    for regla in CONFIG["router"]:
        if any(k in p for k in regla["claves"]):
            return regla["campos"]
    return None


def buscar(pregunta, k=None, campos=None, filtrar=True):
    k = k or CONFIG["k"]
    campos = campos or (detectar_campos(pregunta) if filtrar else None)
    kw = {"query_embeddings": [vec_query(pregunta)], "n_results": k,
          "include": ["documents", "metadatas", "distances"]}
    if campos:
        # Chroma acepta $in para filtrar por varios valores de un metadato
        kw["where"] = ({"campo": campos[0]} if len(campos) == 1
                       else {"campo": {"$in": campos}})
    r = col.query(**kw)
    return [{"texto": r["documents"][0][i],
             "dci": r["metadatas"][0][i]["dci"],
             "campo": r["metadatas"][0][i]["campo"],
             "marca": r["metadatas"][0][i].get("marca", ""),
             "distancia": round(float(r["distances"][0][i]), 4),
             "similitud": round(1 - float(r["distances"][0][i]), 4)}
            for i in range(len(r["documents"][0]))], campo


PRUEBAS = [
    "¿Puedo tomar ibuprofeno con alcohol?",
    "¿Para qué sirve el omeprazol?",
    "Estoy embarazada, ¿puedo tomar paracetamol?",
    "¿Qué efectos secundarios tiene la sertralina?",
]

for q in PRUEBAS:
    campos = detectar_campos(q)
    con, _ = buscar(q, k=3, filtrar=True)
    sin, _ = buscar(q, k=3, filtrar=False)
    print(f"\n{q}")
    print(f"   router -> campos: {campos}")
    print(f"   CON filtro : {[(f['dci'][:12], f['campo'][:22], f['similitud']) for f in con]}")
    print(f"   SIN filtro : {[(f['dci'][:12], f['campo'][:22], f['similitud']) for f in sin]}")

print("""
  QUÉ MIRAR

  Sin filtro, la búsqueda mezcla campos: una pregunta sobre alcohol puede
  traer indicaciones y dosis, que hablan del mismo medicamento pero no
  responden lo que se preguntó.

  Con filtro, todos los fragmentos son del campo correcto y el prompt queda
  limpio. Menos ruido en el contexto es menos oportunidad de que el modelo
  se distraiga: la falla de GENERACIÓN que vimos en teoría.

  La celda 9 pone un número a esta diferencia.
""")


¿Puedo tomar ibuprofeno con alcohol?
   router -> campos: ['drug_interactions', 'warnings']
   CON filtro : [('ibuprofen', 'warnings', 0.8292), ('ibuprofen', 'warnings', 0.8229), ('ibuprofen', 'warnings', 0.8226)]
   SIN filtro : [('ibuprofen', 'warnings', 0.8292), ('ketorolac', 'contraindications', 0.828), ('ibuprofen', 'warnings', 0.8229)]

¿Para qué sirve el omeprazol?
   router -> campos: ['indications_and_usage']
   CON filtro : [('alprazolam', 'indications_and_usage', 0.8066), ('alprazolam', 'indications_and_usage', 0.8066), ('fluoxetine', 'indications_and_usage', 0.8049)]
   SIN filtro : [('clopidogrel', 'drug_interactions', 0.8121), ('clopidogrel', 'drug_interactions', 0.8113), ('alprazolam', 'indications_and_usage', 0.8066)]

Estoy embarazada, ¿puedo tomar paracetamol?
   router -> campos: ['pregnancy', 'warnings']
   CON filtro : [('ketorolac', 'pregnancy', 0.83), ('captopril', 'warnings', 0.827), ('captopril', 'pregnancy', 0.8266)]
   SIN filtro : [('ketorolac', 'pregnancy'

In [21]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 8 — GENERACIÓN CON DEEPSEEK
#
#  Tres reglas no negociables en este dominio:
#   1. Solo responde desde el contexto, citando MEDICAMENTO y SECCIÓN.
#   2. Si no está en el contexto, lo dice. En dosis, JAMÁS calcula.
#   3. Aviso permanente de la fuente y sus límites.
# ═══════════════════════════════════════════════════════════════════════════
from openai import OpenAI

cliente_llm = OpenAI(api_key=DEEPSEEK_KEY,
                     base_url=CONFIG["base_url"]) if DEEPSEEK_KEY else None

SYSTEM_PROMPT = """Eres un asistente que consulta etiquetas de medicamentos publicadas por la FDA de Estados Unidos. Las etiquetas estan en ingles; tu respondes en espanol.

REGLAS ESTRICTAS:
1. Responde UNICAMENTE con informacion del CONTEXTO entregado.
2. Cita SIEMPRE el medicamento y la seccion de la etiqueta de donde sacaste el dato,
   asi: (ibuprofen, seccion warnings).
3. Si el contexto no contiene la respuesta, responde exactamente:
   "Eso no aparece en las etiquetas que tengo cargadas."
   NO completes con conocimiento general.
4. NUNCA calcules ni estimes una dosis que no este escrita literalmente en el contexto,
   en especial dosis pediatricas o por peso.
5. Cierra siempre recordando que es informacion de la FDA de Estados Unidos, que no
   corresponde al registro sanitario peruano y que no sustituye la indicacion de un
   profesional de la salud.
6. Maximo 6 oraciones."""

SIN_RESPUESTA = "Eso no aparece en las etiquetas que tengo cargadas."
LOG = []


def costo_usd(tin, tout, tcache=0):
    p = CONFIG["precios"]
    return ((max(tin - tcache, 0)/1e6)*p["in"] + (tcache/1e6)*p["in_cache"]
            + (tout/1e6)*p["out"])


def responder(pregunta, k=None, temperature=None, filtrar=True):
    k = k or CONFIG["k"]
    temperature = CONFIG["temperature"] if temperature is None else temperature
    t0 = time.time()
    fuentes, campos = buscar(pregunta, k, filtrar=filtrar)
    base = {"pregunta": pregunta, "fuentes": fuentes, "campos": campos,
            "tokens_in": 0, "tokens_cache": 0, "tokens_out": 0, "costo_usd": 0.0}

    mejor = fuentes[0]["similitud"] if fuentes else 0.0
    if mejor < CONFIG["umbral_similitud"]:
        return {**base, "respuesta": SIN_RESPUESTA, "abstuvo": True,
                "motivo": f"similitud máxima {mejor} < umbral {CONFIG['umbral_similitud']}",
                "latencia_s": round(time.time()-t0, 3)}

    if cliente_llm is None:
        return {**base, "respuesta": "(sin llave: no se llamó al modelo)",
                "abstuvo": False, "motivo": "falta DEEPSEEK_API_KEY",
                "latencia_s": round(time.time()-t0, 3)}

    contexto = "\n\n".join(
        f"[Fragmento {i+1} | medicamento: {f['dci']} | seccion: {f['campo']}]\n{f['texto']}"
        for i, f in enumerate(fuentes))
    try:
        r = cliente_llm.chat.completions.create(
            model=CONFIG["modelo_llm"],
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content":
                       f"CONTEXTO (etiquetas openFDA):\n{contexto}\n\nPREGUNTA: {pregunta}"}],
            temperature=temperature, max_tokens=CONFIG["max_tokens"])
        texto = r.choices[0].message.content
        tin = getattr(r.usage, "prompt_tokens", 0)
        tout = getattr(r.usage, "completion_tokens", 0)
        tcache = getattr(r.usage, "prompt_cache_hit_tokens", 0) or 0
        exito, err = True, ""
    except Exception as e:
        texto, tin, tout, tcache = f"Error: {e}", 0, 0, 0
        exito, err = False, str(e)[:120]

    lat = round(time.time()-t0, 3)
    c = costo_usd(tin, tout, tcache)
    LOG.append({"tokens_in": tin, "tokens_cache": tcache, "tokens_out": tout,
                "costo_usd": c, "latencia_s": lat, "exito": exito, "error": err})
    return {**base, "respuesta": texto,
            "abstuvo": texto.strip().startswith(SIN_RESPUESTA[:22]),
            "motivo": "", "tokens_in": tin, "tokens_cache": tcache,
            "tokens_out": tout, "costo_usd": c, "latencia_s": lat}


r = responder("¿Puedo tomar ibuprofeno con alcohol?")
print("PREGUNTA:", r["pregunta"])
print("campos filtrados:", r["campos"], "\n")
print(r["respuesta"], "\n")
print(f"fuentes  : {[(f['dci'], f['campo']) for f in r['fuentes']]}")
print(f"tokens   : {r['tokens_in']} in (+{r['tokens_cache']} caché) / {r['tokens_out']} out")
print(f"costo    : USD {r['costo_usd']:.8f}")
print(f"latencia : {r['latencia_s']} s")

PREGUNTA: ¿Puedo tomar ibuprofeno con alcohol?
campos filtrados: boxed_warning 

Las etiquetas de ibuprofeno advierten que el riesgo de sangrado estomacal severo es mayor si usted consume 3 o más bebidas alcohólicas cada día mientras usa este producto (ibuprofen, sección warnings). También indican que ese riesgo aumenta si toma más cantidad o por más tiempo de lo indicado, si tiene úlceras o problemas de sangrado estomacal, si es mayor de 60 años o si toma anticoagulantes, esteroides u otros AINE (ibuprofen, sección warnings). Las etiquetas no indican una cantidad segura de alcohol ni recomiendan mezclar ambos, solo señalan ese aumento de riesgo (ibuprofen, sección warnings).

Esta es información de la FDA de Estados Unidos, no corresponde al registro sanitario peruano y no sustituye la indicación de un profesional de la salud. 

fuentes  : [('ibuprofen', 'warnings'), ('ibuprofen', 'warnings'), ('ibuprofen', 'warnings'), ('albuterol', 'drug_interactions'), ('ibuprofen', 'warnings')]
to

In [22]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 9 — EVALUACIÓN
#
#  Recall@k mide la BÚSQUEDA y es gratis: solo usa embeddings locales.
#  La abstención mide el PROMPT y sí consume API.
#
#  Novedad de este proyecto: Recall@k POR CAMPO. Vas a ver que el sistema es
#  más débil justo donde el dato escasea, y eso ya lo predijo la celda 4.
#
#  AVISO: este set es una SEMILLA. Tras la primera corrida audita cada fila
#  contra las etiquetas reales. Un set mal etiquetado te hace optimizar hacia
#  el lugar equivocado.
# ═══════════════════════════════════════════════════════════════════════════
CASOS = [
    # (pregunta, tipo, dci esperado, campo esperado)
    ("¿Puedo tomar ibuprofeno con alcohol?",              "dominio", "ibuprofen",    "warnings"),
    ("¿Para qué sirve el omeprazol?",                     "dominio", "omeprazole",   "indications_and_usage"),
    ("¿Cada cuántas horas se toma el paracetamol?",       "dominio", "acetaminophen","dosage_and_administration"),
    ("¿Qué efectos secundarios tiene la sertralina?",     "dominio", "sertraline",   "adverse_reactions"),
    ("¿Quién no debe tomar metformina?",                  "dominio", "metformin",    "contraindications"),
    ("¿La warfarina interactúa con otros medicamentos?",  "dominio", "warfarin",     "drug_interactions"),
    ("¿Para qué sirve el losartán?",                      "dominio", "losartan",     "indications_and_usage"),
    ("¿Qué advertencias tiene el tramadol?",              "dominio", "tramadol",     "warnings"),
    ("¿Cuál es la dosis de amoxicilina?",                 "dominio", "amoxicillin",  "dosage_and_administration"),
    ("¿La loratadina da sueño?",                          "dominio", "loratadine",   "adverse_reactions"),
    # Trampas: el sistema DEBE abstenerse
    ("¿Cuál es la dosis de ibuprofeno para un niño de 20 kilos?", "trampa", None, None),
    ("¿Cuánto cuesta el omeprazol en Lima?",              "trampa", None, None),
    ("¿Yo puedo tomar este medicamento?",                 "trampa", None, None),
    ("¿Cuál es la capital de Francia?",                   "trampa", None, None),
]
dominio = [c for c in CASOS if c[1] == "dominio"]
trampas = [c for c in CASOS if c[1] == "trampa"]

print("RECALL@k  —  mide la búsqueda, no cuesta nada\n")
resumen = {}
for etiqueta, filtrar in [("SIN filtro", False), ("CON filtro", True)]:
    print(f"--- {etiqueta} ---")
    for k in (1, 3, 5):
        aciertos, por_campo = 0, defaultdict(lambda: [0, 0])
        for preg, _, dci_esp, campo_esp in dominio:
            fu, _ = buscar(preg, k=k, filtrar=filtrar)
            ok = any(f["dci"] == dci_esp and f["campo"] == campo_esp for f in fu)
            aciertos += ok
            por_campo[campo_esp][0] += ok
            por_campo[campo_esp][1] += 1
        resumen[(etiqueta, k)] = aciertos / len(dominio)
        print(f"  Recall@{k} = {aciertos/len(dominio):.2f}  ({aciertos}/{len(dominio)})")
    print("  por campo (k=3):")
    for campo, (ok, tot) in sorted(por_campo.items()):
        print(f"     {campo:<30} {ok}/{tot}")
    print()

print("COMPARACIÓN")
print(f"{'k':<4}{'sin filtro':>12}{'con filtro':>12}{'ganancia':>11}")
for k in (1, 3, 5):
    a, b = resumen[("SIN filtro", k)], resumen[("CON filtro", k)]
    print(f"{k:<4}{a:>12.2f}{b:>12.2f}{b-a:>+11.2f}")

if cliente_llm:
    print("\nABSTENCIÓN  —  mide el prompt, sí consume API\n")
    ok_n = 0
    for preg, _, _, _ in trampas:
        rr = responder(preg)
        ok = bool(rr["abstuvo"])
        ok_n += ok
        print(f"   {'OK   ' if ok else 'FALLA'} {preg[:48]:<50} -> {rr['respuesta'][:50]}")
    print(f"\nTasa de abstención correcta = {ok_n/len(trampas):.2f} ({ok_n}/{len(trampas)})")
    print("""
  La primera trampa es la importante: una dosis pediátrica que la etiqueta
  de adulto no trae. Un sistema mal construido la calcula, con total
  seguridad y bien redactada. Ese es el fallo que hay que mostrar en clase.
""")
else:
    print("\n(sin llave: se omite la prueba de abstención)")

RECALL@k  —  mide la búsqueda, no cuesta nada

--- SIN filtro ---
  Recall@1 = 0.50  (5/10)
  Recall@3 = 0.60  (6/10)
  Recall@5 = 0.60  (6/10)
  por campo (k=3):
     adverse_reactions              1/2
     contraindications              1/1
     dosage_and_administration      1/2
     drug_interactions              1/1
     indications_and_usage          1/2
     warnings                       1/2

--- CON filtro ---
  Recall@1 = 0.60  (6/10)
  Recall@3 = 0.60  (6/10)
  Recall@5 = 0.60  (6/10)
  por campo (k=3):
     adverse_reactions              1/2
     contraindications              1/1
     dosage_and_administration      1/2
     drug_interactions              1/1
     indications_and_usage          1/2
     warnings                       1/2

COMPARACIÓN
k     sin filtro  con filtro   ganancia
1           0.50        0.60      +0.10
3           0.60        0.60      +0.00
5           0.60        0.60      +0.00

ABSTENCIÓN  —  mide el prompt, sí consume API

   OK    ¿Cuál es l

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 12 — EMBEDDINGS LOCAL vs API: tabla de diferencias
#
#  Pegar DESPUÉS de la celda 9 (necesita CASOS, buscar, col, chunks, CONFIG).
#
#  OJO: DeepSeek NO tiene endpoint de embeddings. Su API es solo generación.
#  El lado "API" de esta comparación viene de OpenAI o de Gemini, el que
#  tengas configurado.
#
#  Lo interesante no son las dos columnas de métricas. Es la tabla de
#  DESACUERDOS: dónde uno acierta y el otro falla, y por qué.
# ═══════════════════════════════════════════════════════════════════════════
import importlib.util, subprocess, sys, time, shutil
from pathlib import Path
from collections import defaultdict
import numpy as np

CONFIRMAR_GASTO = True     # ponlo en True cuando hayas visto el estimado

# --- Precios de embeddings. Con fecha, como siempre. -----------------------
CONFIG["precios_emb"] = {
    "verificado_el": "2026-09-15",
    "nota": "Verificar en la pagina oficial del proveedor antes de presupuestar.",
    "openai:text-embedding-3-small": 0.02,     # USD por millon de tokens
    "openai:text-embedding-3-large": 0.13,
    "gemini:gemini-embedding-001": 0.15,
}

# --- Detectar qué proveedor de API hay disponible --------------------------
OPENAI_KEY = llave("OPENAI_API_KEY", obligatoria=True)
GEMINI_KEY = llave("GEMINI_API_KEY")

if OPENAI_KEY:
    PROVEEDOR, MODELO_API = "openai", "text-embedding-3-small"
elif GEMINI_KEY:
    PROVEEDOR, MODELO_API = "gemini", "gemini-embedding-001"
else:
    PROVEEDOR = MODELO_API = None

print("PROVEEDOR DE EMBEDDINGS POR API")
if PROVEEDOR is None:
    print("  Ninguno. Pon OPENAI_API_KEY o GEMINI_API_KEY y vuelve a correr.")
    print("  Recuerda: DeepSeek no ofrece embeddings, solo generacion.")
else:
    print(f"  {PROVEEDOR} -> {MODELO_API}")

# --- Cliente de embeddings por API ----------------------------------------
def _instalar(paquete, modulo):
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete])


def vec_api(textos, es_consulta=False):
    """Embeddings por API, normalizados.

    Fijate en una diferencia que se pasa por alto: cada modelo tiene su
    PROPIO protocolo para distinguir pregunta de pasaje.
      e5      -> prefijos de texto  "query: " / "passage: "
      Gemini  -> parametro task_type RETRIEVAL_QUERY / RETRIEVAL_DOCUMENT
      OpenAI  -> no distingue: el mismo modelo para los dos
    Aplicar los prefijos de e5 a OpenAI seria un error silencioso.
    """
    if PROVEEDOR == "openai":
        _instalar("openai", "openai")
        from openai import OpenAI
        cli = OpenAI(api_key=OPENAI_KEY)
        salida = []
        for i in range(0, len(textos), 96):
            r = cli.embeddings.create(model=MODELO_API, input=textos[i:i + 96])
            salida += [d.embedding for d in r.data]
    elif PROVEEDOR == "gemini":
        _instalar("google-genai", "google.genai")
        from google import genai
        from google.genai import types
        cli = genai.Client(api_key=GEMINI_KEY)
        tarea = "RETRIEVAL_QUERY" if es_consulta else "RETRIEVAL_DOCUMENT"
        salida = []
        for i in range(0, len(textos), 64):
            r = cli.models.embed_content(
                model=MODELO_API, contents=textos[i:i + 64],
                config=types.EmbedContentConfig(task_type=tarea))
            salida += [e.values for e in r.embeddings]
            time.sleep(0.2)
    else:
        raise RuntimeError("sin proveedor de API")

    v = np.asarray(salida, dtype=float)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)   # normalizar siempre
    return v.tolist()


# --- Estimar el gasto ANTES de gastarlo -----------------------------------
tokens_est = sum(len(c["texto"]) for c in chunks) / 4      # ingles: ~4 char/token
precio = CONFIG["precios_emb"].get(f"{PROVEEDOR}:{MODELO_API}", 0.0)
costo_est = tokens_est / 1e6 * precio

print(f"\nESTIMADO DE INDEXADO POR API")
print(f"  chunks           : {len(chunks):,}")
print(f"  tokens estimados : {tokens_est:,.0f}")
print(f"  precio           : USD {precio} por millon")
print(f"  costo estimado   : USD {costo_est:.4f}")
print(f"  costo del indice local: USD 0.00")
print(f"\n  CONFIRMAR_GASTO = {CONFIRMAR_GASTO}")

# --- Construir el segundo indice ------------------------------------------
puede = PROVEEDOR is not None and CONFIRMAR_GASTO
col_api = None
t_index_api = None

if puede:
    ruta_api = CONFIG["chroma_path"] + "_api"
    col_api = cliente.get_or_create_collection(
        name=CONFIG["coleccion"] + "_api", metadata={"hnsw:space": "cosine"})
    ya = col_api.count()
    if ya < len(chunks):
        print(f"\nIndexando con {MODELO_API} desde el chunk {ya}...")
        t0 = time.time()
        for i in range(ya, len(chunks), 64):
            bloque = chunks[i:i + 64]
            textos = [c["texto"] for c in bloque]
            col_api.add(ids=[c["id"] for c in bloque], documents=textos,
                        embeddings=vec_api(textos), metadatas=[c["meta"] for c in bloque])
        t_index_api = time.time() - t0
        print(f"  listo en {t_index_api:.1f} s")
    else:
        print(f"\nEl indice API ya estaba completo ({ya} chunks).")


def buscar_api(pregunta, k=5, campos=None):
    r = col_api.query(query_embeddings=[vec_api([pregunta], es_consulta=True)[0]],
                      n_results=k, include=["metadatas", "distances"],
                      **({"where": ({"campo": campos[0]} if len(campos) == 1
                                    else {"campo": {"$in": campos}})} if campos else {}))
    return [{"dci": r["metadatas"][0][i]["dci"], "campo": r["metadatas"][0][i]["campo"],
             "similitud": round(1 - float(r["distances"][0][i]), 4)}
            for i in range(len(r["metadatas"][0]))]


# --- Metricas comparables --------------------------------------------------
def tam_mb(ruta):
    p = Path(ruta)
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6 if p.exists() else 0


def evaluar(fn_buscar, casos, ks=(1, 3, 5)):
    """Recall@k, MRR y latencia media. Solo recuperacion: no llama al generador."""
    recall = {k: 0 for k in ks}
    rr, tiempos, rankings = [], [], {}
    for preg, _, dci_esp, campo_esp in casos:
        t0 = time.time()
        top = fn_buscar(preg, 10)
        tiempos.append((time.time() - t0) * 1000)
        pos = next((i + 1 for i, f in enumerate(top)
                    if f["dci"] == dci_esp and f["campo"] == campo_esp), None)
        rr.append(1 / pos if pos else 0.0)
        rankings[preg] = top
        for k in ks:
            recall[k] += int(pos is not None and pos <= k)
    n = len(casos)
    return ({f"Recall@{k}": round(recall[k] / n, 2) for k in ks}
            | {"MRR": round(sum(rr) / n, 3),
               "ms por consulta": round(sum(tiempos) / n)}), rankings


dominio = [c for c in CASOS if c[1] == "dominio"]
buscar_local = lambda q, k: buscar(q, k=k, filtrar=False)[0]

m_local, rank_local = evaluar(buscar_local, dominio)
m_local |= {"modelo": CONFIG["modelo_emb"], "dimensiones": DIM,
            "costo indexado USD": 0.0, "disco MB": round(tam_mb(CONFIG["chroma_path"]), 1),
            "offline": "si", "s indexado": "medido en celda 6"}

filas = {"LOCAL": m_local}

if puede:
    m_api, rank_api = evaluar(lambda q, k: buscar_api(q, k), dominio)
    m_api |= {"modelo": MODELO_API, "dimensiones": len(vec_api(["x"])[0]),
              "costo indexado USD": round(costo_est, 4),
              "disco MB": round(tam_mb(CONFIG["chroma_path"]), 1),
              "offline": "no",
              "s indexado": round(t_index_api, 1) if t_index_api else "en cache"}
    filas["API"] = m_api

orden = ["modelo", "dimensiones", "Recall@1", "Recall@3", "Recall@5", "MRR",
         "ms por consulta", "costo indexado USD", "disco MB", "offline"]

print(f"\n{'='*78}\nTABLA COMPARATIVA\n{'='*78}")
anchos = max(len(k) for k in orden) + 2
cab = "".join(f"{n:>22}" for n in filas)
print(f"{'metrica':<{anchos}}{cab}")
print("-" * (anchos + 22 * len(filas)))
for m in orden:
    vals = "".join(f"{str(filas[n].get(m, '-')):>22}" for n in filas)
    print(f"{m:<{anchos}}{vals}")

# --- LO INTERESANTE: donde DISCREPAN --------------------------------------
if puede:
    def jaccard(a, b):
        A = {(f["dci"], f["campo"]) for f in a[:3]}
        B = {(f["dci"], f["campo"]) for f in b[:3]}
        return len(A & B) / len(A | B) if A | B else 0.0

    distinto_top1, jac, discordantes = 0, [], []
    for preg, _, dci_esp, campo_esp in dominio:
        L, A = rank_local[preg], rank_api[preg]
        t1L = (L[0]["dci"], L[0]["campo"]) if L else None
        t1A = (A[0]["dci"], A[0]["campo"]) if A else None
        distinto_top1 += int(t1L != t1A)
        jac.append(jaccard(L, A))
        okL = any(f["dci"] == dci_esp and f["campo"] == campo_esp for f in L[:3])
        okA = any(f["dci"] == dci_esp and f["campo"] == campo_esp for f in A[:3])
        if okL != okA:
            discordantes.append((preg, "LOCAL" if okL else "API", t1L, t1A))

    n = len(dominio)
    print(f"\n{'='*78}\nANALISIS DE DESACUERDO\n{'='*78}")
    print(f"  consultas donde el top-1 difiere : {distinto_top1}/{n} "
          f"({distinto_top1/n*100:.0f}%)")
    print(f"  solapamiento medio del top-3     : {sum(jac)/n:.2f}  (1.00 = identicos)")
    print(f"  casos donde uno acierta y el otro falla: {len(discordantes)}")

    if discordantes:
        print(f"\n  {'consulta':<44}{'gana':<8}{'top-1 local':<26}{'top-1 api'}")
        print("  " + "-" * 104)
        for preg, gana, t1L, t1A in discordantes:
            print(f"  {preg[:42]:<44}{gana:<8}{str(t1L)[:24]:<26}{str(t1A)[:24]}")

    print("""
  COMO SE LEE ESTO

  Si el solapamiento del top-3 es alto y los Recall casi empatan, la
  conclusion es economica: usa el local, es gratis y funciona offline.

  Si el API gana de forma consistente, mira DONDE gana. Casi siempre es en
  las consultas mas largas o con vocabulario tecnico en ingles, y casi
  nunca en las consultas cortas y coloquiales.

  Y ojo con las escalas: las similitudes de los dos modelos NO son
  comparables entre si. Un umbral calibrado para uno no sirve para el otro.
  Por eso la tabla compara ORDEN (Recall, MRR), no numeros absolutos.

  Un empate tambien es un resultado, y es el mas util: significa que la
  decision la toma el costo, no la calidad.
""")
else:
    print("\n(sin proveedor de API o CONFIRMAR_GASTO=False: solo se muestra el local)")

PROVEEDOR DE EMBEDDINGS POR API
  openai -> text-embedding-3-small

ESTIMADO DE INDEXADO POR API
  chunks           : 1,572
  tokens estimados : 397,232
  precio           : USD 0.02 por millon
  costo estimado   : USD 0.0079
  costo del indice local: USD 0.00

  CONFIRMAR_GASTO = True

Indexando con text-embedding-3-small desde el chunk 0...


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 11 — INTERFAZ  (chunks, temperature, costo y filtro por campo)
#
#  Pegar DESPUÉS de la celda 8. Es otro ADAPTADOR sobre el mismo motor:
#  no contiene lógica de RAG, solo llama a responder().
# ═══════════════════════════════════════════════════════════════════════════
import importlib.util, subprocess, sys
if importlib.util.find_spec("ipywidgets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])

import ipywidgets as W
from IPython.display import display, HTML, clear_output

# --- Permite forzar un campo desde el desplegable --------------------------
# buscar() resuelve detectar_campos por nombre global, así que re-enlazarlo
# aquí basta. Guardamos el original para no envolverlo dos veces.
if "_detectar_original" not in globals():
    _detectar_original = detectar_campos

CAMPO_FORZADO = None

def detectar_campos(pregunta):
    if CAMPO_FORZADO:
        return [CAMPO_FORZADO]
    return _detectar_original(pregunta)

SESION = {"consultas": 0, "tokens_in": 0, "tokens_cache": 0,
          "tokens_out": 0, "costo": 0.0, "abstenciones": 0}

# ---------------------------------------------------------------- controles
est = {"description_width": "110px"}
ancho = W.Layout(width="350px")

sl_k = W.IntSlider(value=CONFIG["k"], min=1, max=15, step=1,
                   description="chunks (k):", continuous_update=False,
                   style=est, layout=ancho)

sl_temp = W.FloatSlider(value=CONFIG["temperature"], min=0.0, max=1.5, step=0.1,
                        description="temperature:", readout_format=".1f",
                        continuous_update=False, style=est, layout=ancho)

sl_umbral = W.FloatSlider(value=CONFIG["umbral_similitud"], min=0.50, max=0.95,
                          step=0.01, description="umbral:", readout_format=".2f",
                          continuous_update=False, style=est, layout=ancho)

ck_filtro = W.Checkbox(value=True, description="filtrar por campo",
                       indent=False, layout=W.Layout(width="200px"))

dd_campo = W.Dropdown(options=["auto (router)"] + CONFIG["campos"],
                      value="auto (router)", description="campo:",
                      style=est, layout=ancho)

txt = W.Textarea(placeholder="Ej: ¿puedo tomar ibuprofeno con alcohol?",
                 layout=W.Layout(width="99%", height="70px"))

btn = W.Button(description="Consultar", button_style="primary",
               icon="search", layout=W.Layout(width="150px"))

medidor = W.HTML()
out = W.Output()

AVISO = ("Etiquetas de la FDA de Estados Unidos, en inglés, derivadas de documentos SPL. "
         "No corresponden al registro sanitario peruano ni sustituyen la indicación "
         "de un profesional de la salud.")

# ---------------------------------------------------------------- utilidades
def pintar_medidor():
    s = SESION
    cache_pct = s["tokens_cache"] / s["tokens_in"] * 100 if s["tokens_in"] else 0
    medidor.value = (
        "<div style='display:flex;gap:24px;flex-wrap:wrap;padding:10px 4px;"
        "border-bottom:1px solid rgba(128,128,128,.35);font-size:13px'>"
        f"<div><b>{s['consultas']}</b><br><span style='opacity:.65'>consultas</span></div>"
        f"<div><b>{s['abstenciones']}</b><br><span style='opacity:.65'>abstenciones</span></div>"
        f"<div><b>{s['tokens_in']:,}</b><br><span style='opacity:.65'>tokens in</span></div>"
        f"<div><b>{cache_pct:.0f}%</b><br><span style='opacity:.65'>de caché</span></div>"
        f"<div><b>{s['tokens_out']:,}</b><br><span style='opacity:.65'>tokens out</span></div>"
        f"<div><b>USD {s['costo']:.6f}</b><br><span style='opacity:.65'>costo sesión</span></div>"
        f"<div><b>{col.count():,}</b><br><span style='opacity:.65'>chunks indexados</span></div>"
        "</div>")


def esc(t):
    return (str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))


def consultar(_=None):
    global CAMPO_FORZADO
    pregunta = txt.value.strip()
    with out:
        clear_output(wait=True)
        if not pregunta:
            print("Escribe una pregunta.")
            return

        # Los controles escriben en CONFIG: una sola fuente de verdad.
        CONFIG["umbral_similitud"] = sl_umbral.value
        CAMPO_FORZADO = None if dd_campo.value == "auto (router)" else dd_campo.value

        display(HTML("<i style='opacity:.6'>buscando en las etiquetas…</i>"))
        r = responder(pregunta, k=sl_k.value, temperature=sl_temp.value,
                      filtrar=ck_filtro.value)
        clear_output(wait=True)

        SESION["consultas"] += 1
        SESION["tokens_in"] += r["tokens_in"]
        SESION["tokens_cache"] += r["tokens_cache"]
        SESION["tokens_out"] += r["tokens_out"]
        SESION["costo"] += r["costo_usd"]
        SESION["abstenciones"] += bool(r["abstuvo"])
        pintar_medidor()

        # --- respuesta ---
        borde = "#c9a227" if r["abstuvo"] else "#2e8b57"
        etiqueta = "SE ABSTUVO" if r["abstuvo"] else "RESPUESTA"
        motivo = (f"<div style='font-size:12px;opacity:.7;margin-top:8px'>{esc(r['motivo'])}</div>"
                  if r.get("motivo") else "")
        display(HTML(
            f"<div style='border-left:4px solid {borde};padding:10px 14px;margin:10px 0'>"
            f"<div style='font-size:11px;letter-spacing:.08em;opacity:.6'>{etiqueta}</div>"
            f"<div style='margin-top:6px;line-height:1.55'>"
            f"{esc(r['respuesta']).replace(chr(10), '<br>')}</div>{motivo}</div>"))

        # --- métricas de la consulta ---
        campos = r["campos"] or ["(sin filtro)"]
        display(HTML(
            "<div style='display:flex;gap:20px;flex-wrap:wrap;font-size:13px;margin:6px 0'>"
            f"<div><span style='opacity:.65'>campos</span> <b>{esc(', '.join(campos))}</b></div>"
            f"<div><b>{r['tokens_in']}</b> <span style='opacity:.65'>in</span></div>"
            f"<div><b>{r['tokens_cache']}</b> <span style='opacity:.65'>caché</span></div>"
            f"<div><b>{r['tokens_out']}</b> <span style='opacity:.65'>out</span></div>"
            f"<div><b>USD {r['costo_usd']:.8f}</b></div>"
            f"<div><b>{r['latencia_s']}s</b></div></div>"))

        # --- los chunks recuperados ---
        filas = "".join(
            "<tr>"
            f"<td style='padding:4px 9px'>{i}</td>"
            f"<td style='padding:4px 9px'><b>{esc(f['dci'])}</b></td>"
            f"<td style='padding:4px 9px'>{esc(f['campo'])}</td>"
            f"<td style='padding:4px 9px'>{esc(f.get('marca',''))[:18]}</td>"
            f"<td style='padding:4px 9px'>{f['similitud']:.4f}</td>"
            f"<td style='padding:4px 9px;opacity:.6'>{f['distancia']:.4f}</td>"
            f"<td style='padding:4px 9px;font-size:12px'>{esc(f['texto'][:150])}…</td></tr>"
            for i, f in enumerate(r["fuentes"], 1))
        display(HTML(
            f"<details open><summary style='cursor:pointer;font-size:13px'>"
            f"Chunks recuperados ({len(r['fuentes'])})</summary>"
            "<table style='border-collapse:collapse;font-size:13px;margin-top:8px'>"
            "<tr style='opacity:.65;text-align:left'>"
            "<th style='padding:4px 9px'>#</th><th style='padding:4px 9px'>medicamento</th>"
            "<th style='padding:4px 9px'>campo</th><th style='padding:4px 9px'>marca</th>"
            "<th style='padding:4px 9px'>similitud</th><th style='padding:4px 9px'>distancia</th>"
            "<th style='padding:4px 9px'>texto</th></tr>"
            f"{filas}</table></details>"))


def preset(q, campo="auto (router)"):
    def _(_b):
        txt.value = q
        dd_campo.value = campo
        consultar()
    return _


btn.on_click(consultar)

atajos = W.HBox([
    W.Button(description="Alcohol", layout=W.Layout(width="120px")),
    W.Button(description="Para qué sirve", layout=W.Layout(width="140px")),
    W.Button(description="Trampa: dosis niño", layout=W.Layout(width="165px")),
])
atajos.children[0].on_click(preset("¿Puedo tomar ibuprofeno con alcohol?"))
atajos.children[1].on_click(preset("¿Para qué sirve el omeprazol?"))
atajos.children[2].on_click(
    preset("¿Cuál es la dosis de ibuprofeno para un niño de 20 kilos?"))

pintar_medidor()
display(W.VBox([
    W.HTML("<h3 style='margin:0'>Consulta de etiquetas de medicamentos</h3>"
           f"<div style='opacity:.65;font-size:12.5px;margin-top:4px'>{AVISO}</div>"),
    medidor,
    W.HBox([W.VBox([sl_k, sl_temp]), W.VBox([sl_umbral, dd_campo, ck_filtro])]),
    txt,
    W.HBox([btn, atajos]),
    out,
]))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELDA 10 — DEMO Y CIERRE
# ═══════════════════════════════════════════════════════════════════════════
DEMO = [
    "¿Puedo tomar ibuprofeno con alcohol?",                        # warnings
    "¿Para qué sirve el omeprazol?",                               # indications
    "¿Cuál es la dosis de ibuprofeno para un niño de 20 kilos?",    # debe abstenerse
]

for q in DEMO:
    r = responder(q)
    marca = "[SE ABSTUVO] " if r["abstuvo"] else ""
    print(f"\n{'='*72}\nP: {q}")
    print(f"   campos filtrados: {r['campos']}")
    print(f"R: {marca}{r['respuesta'][:420]}")
    if not r["abstuvo"] and r["fuentes"]:
        print(f"   fuentes {[(f['dci'], f['campo']) for f in r['fuentes'][:3]]}")
        print(f"   {r['tokens_in']}+{r['tokens_out']} tokens | "
              f"USD {r['costo_usd']:.8f} | {r['latencia_s']}s")

if LOG:
    ok = [l for l in LOG if l["exito"]]
    tin = sum(l["tokens_in"] for l in ok); tca = sum(l["tokens_cache"] for l in ok)
    tou = sum(l["tokens_out"] for l in ok); tot = sum(l["costo_usd"] for l in ok)
    lat = [l["latencia_s"] for l in ok]
    print(f"\n{'='*72}\nCONTABILIDAD DE LA SESIÓN")
    print(f"  llamadas exitosas : {len(ok)} de {len(LOG)}")
    print(f"  tokens entrada    : {tin:,} (de caché {tca:,} = {tca/max(tin,1)*100:.0f}%)")
    print(f"  tokens salida     : {tou:,}")
    print(f"  costo total       : USD {tot:.6f}")
    print(f"  costo por consulta: USD {tot/len(ok):.6f}")
    print(f"  latencia media    : {sum(lat)/len(lat):.2f} s")
    print(f"  precios verificados el {CONFIG['precios']['verificado_el']}")

print("""
========================================================================
LO QUE TE LLEVAS

1. Un corpus que ya viene troceado por quien lo escribió cambia el
   pipeline: el chunking deja de ser "cuántos caracteres" y pasa a ser
   "qué campo".
2. El informe de cobertura va ANTES de chunkear. Define qué preguntas el
   sistema tiene derecho a contestar.
3. La metadata convierte una búsqueda en siete. El filtro por campo se
   mide, no se argumenta.
4. Recall POR CAMPO revela dónde el sistema es débil, y coincide con las
   filas escasas de la tabla de cobertura.
5. En dosis, el sistema NO calcula. Extrae o se abstiene. Un número
   inventado suena igual de seguro que uno correcto.
6. La fuente y sus límites van en la respuesta, no al pie.

========================================================================
TU PROYECTO

Aplica este pipeline a un corpus de tu dominio que venga con campos:

  [ ] Diagrama Mermaid en el README, offline separado de online
  [ ] config sin un solo número dentro del código
  [ ] Informe de cobertura de campos antes del chunking
  [ ] Campo como metadata, y filtro por campo medido contra el baseline
  [ ] 30 preguntas de evaluación, 5 de ellas trampa
  [ ] Recall@k global Y por campo
  [ ] Log de costos con una fila por llamada
  [ ] Streamlit local funcionando

  La pregunta de la sustentación:
  "¿Tu RAG falla por recuperación o por generación?
   Muéstrame el número que lo prueba."

========================================================================
FUENTE: openFDA, U.S. Food and Drug Administration.
Etiquetas en ingles, derivadas de documentos SPL. NO corresponden al
registro sanitario peruano ni sustituyen la indicacion de un profesional.
========================================================================
""")